# Curio Seeker - SC14

## Import packages

In [6]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [7]:
import harpy as hp
import spatialdata as sd
import pandas as pd
import numpy as np
import os
import geopandas as gpd
import matplotlib
from matplotlib import pyplot as plt
import gc
import spatialdata_plot
from spatialdata.transformations import get_transformation, remove_transformation, set_transformation

matplotlib.use("Agg")

import logging
logging.getLogger('ome_zarr').setLevel(logging.ERROR) # This mutes unnecessary Zarr warning (see: https://github.com/scverse/spatialdata/issues/400)

In [8]:
import sys
sys.path.append(os.path.dirname(os.path.dirname(os.path.abspath(''))))
from utils.in_tissue import add_in_tissue, assign_region
from utils.plotting import best_datatree_scale, rel_fontsize, glasbey_cmap
from utils.qc import qc_metrics
from _curio import curio

## Create sdata

In [9]:
dataset_id = 'SC14'

input_path = f'/data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/Curio_seeker/{dataset_id}/curioseeker/results/OUTPUT/{dataset_id}'
output_path = f'/data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/Curio_seeker/{dataset_id}/harpy/sdata.zarr'
barcode_decoding_file = f'/data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/raw/Curio_seeker/barcode_decoding_files/A0080_056_BeadBarcodes.txt'

pipeline_name = 'CurioSeeker'
species = 'human'

In [10]:
# sdata = sd.read_zarr(output_path)
# sdata

In [11]:
sdata = curio(
    path=input_path, 
    dataset_id=dataset_id,
    output=output_path,
)
sdata

/data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/scripts/2024-07-UTBenchmark-SpC/Curio_seeker/harpy/spatialdata_curio.py:63: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 char and different from '\s+' are interpreted as regex); you can avoid this warning by specifying engine='python'.
  metrics = pd.read_csv(path / file_names[CurioKeys.METRICS_FILE], sep=r"\,", header=0)
2026-01-28 12:07:12.109 | INFO     | harpy.image._manager:add_layer:47 - Writing results to layer 'SC14_labels'


SpatialData object, with associated Zarr store: /data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/Curio_seeker/SC14/harpy/sdata.zarr
├── Labels
│     └── 'SC14_labels': DataArray[yx] (4757, 4710)
├── Shapes
│     └── 'SC14': GeoDataFrame shape: (26901, 1) (2D shapes)
└── Tables
      └── 'table': AnnData (26901, 23297)
with coordinate systems:
    ▸ 'SC14', with elements:
        SC14_labels (Labels), SC14 (Shapes)

In [12]:
levels = [*sdata.tables]
print(f"{pipeline_name} levels: {levels}")

CurioSeeker levels: ['table']


In [13]:
# Assert whether number of shapes matches number of label IDs
import dask.array as da

for level in levels:
    number_of_shapes = len(sdata.shapes[dataset_id])
    number_of_label_IDs = len(da.unique(sdata.labels[f"{dataset_id}_labels"].data).compute()) - 1
    assert number_of_shapes == number_of_label_IDs, f"{level}: Number of shapes ({number_of_shapes}) does not match number of label IDs ({number_of_label_IDs})"

## Calculate pixel size

In [14]:
# Determine pixel size in microns
from math import dist

union_geom = sdata.shapes[dataset_id].union_all()
bbox = union_geom.minimum_rotated_rectangle

sdata = hp.sh.add_shapes_layer(
    sdata,
    input = gpd.GeoDataFrame(geometry=[bbox]),
    output_layer = f'{dataset_id}_capture_area_bbox',
    transformations = get_transformation(sdata.shapes[dataset_id], get_all=True),
    overwrite = True,
)

coords = list(bbox.exterior.coords)

# Compute lengths of bbox
side1 = dist(coords[0], coords[1])
side2 = dist(coords[1], coords[2])

microns_per_pixel_x = 3000 / side1
microns_per_pixel_y = 3000 / side2

print(f"Pixel size: {microns_per_pixel_x:.3f} µm (X), {microns_per_pixel_y:.3f} µm (Y)")

microns_per_pixel = (microns_per_pixel_x + microns_per_pixel_y) / 2
print(f"Pixel size: {microns_per_pixel:.3f} µm")

# NOTE: This is not a great way to have to surmise what the 'pixel size' is, but it should be good enough. This could be off if the sides of the capture area are not exactly 3 mm.

Pixel size: 0.636 µm (X), 0.633 µm (Y)
Pixel size: 0.635 µm


## Hexagonal tesselation

In [15]:
custom_bins = ['hexagonal_grid_20um', 'hexagonal_grid_120um'] # Names refer to the diagonal of the hexagon

In [16]:
min_x, min_y, max_x, max_y = sdata.shapes[dataset_id]['geometry'].total_bounds
shape = (max_y - min_y, max_x - min_x)

for custom_bin in custom_bins:
    gc.collect()
    
    size = (int(custom_bin.split("hexagonal_grid_")[1].split("um")[0])/2) / microns_per_pixel # Radius of the hexagon in pixels
    
    # Create hexagonal grids
    sdata = hp.im.add_grid_labels_layer(
        sdata, 
        shape=shape,
        size=size,
        output_shapes_layer=f"{custom_bin}_shapes", 
        output_labels_layer=f"{custom_bin}_labels", 
        grid_type="hexagon", 
        offset=(0, 0), 
        chunks=2048, 
        client=None, 
        transformations=get_transformation(sdata.shapes[dataset_id], get_all=True),
        scale_factors=(2, 2, 2, 2),
        overwrite=True
    )

    # Rebin data based on hexagonal grid
    sdata = hp.tb.bin_counts( 
        sdata,
        table_layer="table",
        labels_layer=f"{custom_bin}_labels",
        output_layer=f"{custom_bin}_table",
        append = False,
        to_coordinate_system = dataset_id,
        overwrite=True,
    )

2026-01-28 12:07:29.294 | INFO     | harpy.image._manager:add_layer:47 - Writing results to layer 'hexagonal_grid_20um_labels'
2026-01-28 12:07:30.964 | INFO     | harpy.table._allocation:_aggregate:488 - Calculating cell counts.
2026-01-28 12:07:31.202 | INFO     | harpy.table._allocation:bin_counts:363 - 
 Number of spots per bin  Frequency
                       1      10745
                       2       5758
                       3       1446
                       4         68
                       5          1
2026-01-28 12:07:35.429 | INFO     | harpy.image._manager:add_layer:47 - Writing results to layer 'hexagonal_grid_120um_labels'
2026-01-28 12:07:36.225 | INFO     | harpy.table._allocation:_aggregate:488 - Calculating cell counts.
2026-01-28 12:07:36.446 | INFO     | harpy.table._allocation:bin_counts:363 - 
 Number of spots per bin  Frequency
                       1          4
                       2          9
                       3         12
                     

## Specify plotting parameters

In [17]:
DPI = 300
MAX_DIM_FOR_PLOTS = 10000

figsize_factor = DPI
figsize = (sdata.labels[f'{dataset_id}_labels'].shape[1]/figsize_factor, sdata.labels[f'{dataset_id}_labels'].shape[0]/figsize_factor)
print(f'figsize: {figsize}')

plots_path = os.path.join(os.path.dirname(output_path), 'plots')
os.makedirs(plots_path, exist_ok=True)

print(f'plots_path: {plots_path}')

figsize: (15.7, 15.856666666666667)
plots_path: /data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/Curio_seeker/SC14/harpy/plots


## Expand labels layers for plotting

In [18]:
# Expand labels layer masks for plotting (otherwise, the plots are not very clear)
for level in ['table']:
    
    radius = np.sqrt(sdata.shapes[dataset_id].geometry.iloc[0].area / np.pi)
    
    sdata = hp.im.expand_labels_layer(
        sdata,
        labels_layer=f"{dataset_id}_labels",
        distance=(radius * 1.5) - radius, # Radius increased by 150%
        output_labels_layer=f"{dataset_id}_labels",
        overwrite=True,
)
    
# NOTE: The result of this is that plots using the labels layer do not show spots to scale, but the visualization is improved.


2026-01-28 12:07:42.286 | INFO     | harpy.image.segmentation._map:_combine_dask_arrays:339 - Linking labels across chunks.
2026-01-28 12:07:42.308 | INFO     | harpy.image._manager:add_layer:47 - Writing results to layer 'SC14_labels'
2026-01-28 12:07:43.147 | WARNING  | harpy.utils._io:_incremental_io_on_disk:53 - layer with name 'SC14_labels' already exists. Overwriting...


## Create crop for zoomed-in plots

In [19]:
min_x = sdata.labels[f"{dataset_id}_labels"].shape[0] / 2 # Center of image
max_x = min_x + round(200 / microns_per_pixel) # 200 micrometer
min_y = sdata.labels[f"{dataset_id}_labels"].shape[1] / 2 # Center of image
max_y = min_y + round(200 / microns_per_pixel) # 200 micrometer

sdata_crop = sdata.query.bounding_box(
    min_coordinate=[min_x, min_y], 
    max_coordinate=[max_x, max_y], 
    axes=("x", "y"), 
    target_coordinate_system=dataset_id
)

figsize_crop = (3,3)

## Plot Curio Seeker spots

In [20]:
for level in levels:
    gc.collect()

    level_plots_path = os.path.join(plots_path, level)
    os.makedirs(level_plots_path, exist_ok=True)

    # Plot full capture area
    fig, ax = plt.subplots(figsize=figsize, dpi=DPI)

    title = f"{pipeline_name} - spots"

    sdata.pl.render_shapes(
            element=dataset_id, fill_alpha=0.15, outline_width=1, outline_alpha=1, color='#41b8de', outline_color='#41b8de'
    ).pl.show(
        coordinate_systems=dataset_id,
        title=title,
        figsize=figsize,
        colorbar=False,
        dpi=DPI,
        ax=ax,
        pad_extent=rel_fontsize(fig, 0.05)
    )

    ax.set_title(title, fontsize=rel_fontsize(fig, 0.02))
    ax.set_xlabel("x", fontsize=rel_fontsize(fig, 0.02))
    ax.set_ylabel("y", fontsize=rel_fontsize(fig, 0.02))
    ax.tick_params(axis="both", which="both", labelsize=rel_fontsize(fig, 0.015))

    save_path = os.path.join(level_plots_path, f'{pipeline_name}_spots.png')
    fig.savefig(save_path, bbox_inches="tight", pad_inches=max(fig.get_size_inches())/100, facecolor="white", transparent=False)

    plt.close(fig)

    print("Plot saved at:", save_path, flush=True)

    # Plot crop
    fig, ax = plt.subplots(figsize=figsize_crop, dpi=DPI)

    title = f"{pipeline_name} - spots"

    sdata_crop.pl.render_shapes(
            element=dataset_id, fill_alpha=0.15, outline_width=0.25, outline_alpha=1, color='#41b8de', outline_color='#41b8de', method="matplotlib"
    ).pl.show(
        coordinate_systems=dataset_id,
        title=title,
        figsize=figsize_crop,
        colorbar=False,
        dpi=DPI,
        ax=ax,
    )

    ax.set_title(title, fontsize=rel_fontsize(fig, 0.02))
    ax.set_xlabel("x", fontsize=rel_fontsize(fig, 0.02))
    ax.set_ylabel("y", fontsize=rel_fontsize(fig, 0.02))
    ax.tick_params(axis="both", which="both", labelsize=rel_fontsize(fig, 0.015))

    ax.set_xlim(min_x, max_x)
    ax.set_ylim(min_y, max_y)
    ax.invert_yaxis()

    save_path = os.path.join(level_plots_path, f'crop_{pipeline_name}_spots.png')
    fig.savefig(save_path, bbox_inches="tight", pad_inches=max(fig.get_size_inches())/100, facecolor="white", transparent=False)

    plt.close(fig)

    print("Plot saved at:", save_path, flush=True)
    
# NOTE: Spots are a rough approximation of the correct size.

INFO     Value for parameter 'color' appears to be a color, using it as such.                                      
INFO     Using 'datashader' backend with 'None' as reduction method to speed up plotting. Depending on the         
         reduction method, the value range of the plot might change. Set method to 'matplotlib' to disable this    
         behaviour.                                                                                                
Plot saved at: /data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/Curio_seeker/SC14/harpy/plots/table/CurioSeeker_spots.png
INFO     Value for parameter 'color' appears to be a color, using it as such.                                      
Plot saved at: /data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/Curio_seeker/SC14/harpy/plots/table/crop_CurioSeeker_spots.png


## Plot Curio Seeker columns

In [ ]:
# Plot Curio Seeker numerical columns
for level in levels:
    for column in ['nCount_RNA', 'nFeature_RNA', 'numReads', 'log10_numReads', 'log10_nCount_RNA', 'log10_nFeature_RNA', 'percent.rp', 'percent.mt', 'nCount_SCT', 'nFeature_SCT', 'log_umi']:
        gc.collect()
        
        level_plots_path = os.path.join(plots_path, level)
        os.makedirs(level_plots_path, exist_ok=True)
        
        values = sdata.tables["table"].obs[column].to_numpy()
        vmin, vmax = np.percentile(values, [1, 99])
        norm = matplotlib.colors.Normalize(vmin=vmin, vmax=vmax)    
        
        fig, ax = plt.subplots(figsize=figsize, dpi=DPI)

        title = f"{pipeline_name} - {column}"

        sdata.pl.render_labels(
            element=f"{dataset_id}_labels", fill_alpha=1, contour_px=5, outline_alpha=1, color=column, table_name='table', cmap="viridis", norm=norm
        ).pl.show(
            coordinate_systems=dataset_id,
            title=title,
            figsize=figsize,
            colorbar=True,
            dpi=DPI,
            ax=ax,
            pad_extent=rel_fontsize(fig, 0.05)
        )

        ax.set_title(title, fontsize=rel_fontsize(fig, 0.02))
        ax.set_xlabel("x", fontsize=rel_fontsize(fig, 0.02))
        ax.set_ylabel("y", fontsize=rel_fontsize(fig, 0.02))
        ax.tick_params(axis="both", which="both", labelsize=rel_fontsize(fig, 0.015))
        
        cbar = fig.axes[-1]
        cbar.set_ylabel(column, rotation=270, labelpad=rel_fontsize(fig, 0.025), fontsize = rel_fontsize(fig, 0.02))
        cbar.tick_params(axis="y", which="both", labelsize=rel_fontsize(fig, 0.015))

        save_path = os.path.join(level_plots_path, f'{pipeline_name}_{column}.png')
        fig.savefig(save_path, bbox_inches="tight", pad_inches=max(fig.get_size_inches())/100, facecolor="white", transparent=False)

        plt.close(fig)

        print("Plot saved at:", save_path, flush=True)

Plot saved at: /data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/Curio_seeker/SC14/harpy/plots/table/CurioSeeker_nCount_RNA.png
Plot saved at: /data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/Curio_seeker/SC14/harpy/plots/table/CurioSeeker_nFeature_RNA.png
Plot saved at: /data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/Curio_seeker/SC14/harpy/plots/table/CurioSeeker_numReads.png
Plot saved at: /data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/Curio_seeker/SC14/harpy/plots/table/CurioSeeker_log10_numReads.png
Plot saved at: /data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/Curio_seeker/SC14/harpy/plots/table/CurioSeeker_log10_nCount_RNA.png
Plot saved at: /data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/Curio_seeker/SC14/harpy/plots/table/Cur

In [22]:
# Plot Curio Seeker categorical columns
for level in levels:
    for column in ['SCT_snn_res.0.2', 'seurat_clusters', 'cluster']:
        gc.collect()
        
        level_plots_path = os.path.join(plots_path, level)
        os.makedirs(level_plots_path, exist_ok=True)
        
        cmap = glasbey_cmap(sdata.tables['table'].obs[column].nunique())
        
        with matplotlib.rc_context({'legend.markerscale': 2}):
            fig, ax = plt.subplots(figsize=figsize, dpi=DPI)

            title = f"{pipeline_name} - {column}"

            sdata.pl.render_labels(
                element=f"{dataset_id}_labels", fill_alpha=1, contour_px=5, outline_alpha=1, color=column, table_name='table', cmap=cmap
            ).pl.show(
                coordinate_systems=dataset_id,
                title=title,
                figsize=figsize,
                colorbar=False,
                legend_fontsize=rel_fontsize(fig, 0.015), 
                legend_fontweight='bold', 
                legend_loc='right margin',
                na_in_legend=False,
                dpi=DPI,
                ax=ax,
                pad_extent=rel_fontsize(fig, 0.05)
            )

            ax.set_title(title, fontsize=rel_fontsize(fig, 0.02))
            ax.set_xlabel("x", fontsize=rel_fontsize(fig, 0.02))
            ax.set_ylabel("y", fontsize=rel_fontsize(fig, 0.02))
            ax.tick_params(axis="both", which="both", labelsize=rel_fontsize(fig, 0.015))

            save_path = os.path.join(level_plots_path, f'{pipeline_name}_{column}.png')
            fig.savefig(save_path, bbox_inches="tight", pad_inches=max(fig.get_size_inches())/100, facecolor="white", transparent=False)

            plt.close(fig)

            print("Plot saved at:", save_path)


Plot saved at: /data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/Curio_seeker/SC14/harpy/plots/table/CurioSeeker_SCT_snn_res.0.2.png
Plot saved at: /data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/Curio_seeker/SC14/harpy/plots/table/CurioSeeker_seurat_clusters.png
Plot saved at: /data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/Curio_seeker/SC14/harpy/plots/table/CurioSeeker_cluster.png


## Plot hexagonal spots

In [23]:
for custom_bin in custom_bins:
    gc.collect()

    level_plots_path = os.path.join(plots_path, custom_bin)
    os.makedirs(level_plots_path, exist_ok=True)

    # Plot full image
    fig, ax = plt.subplots(figsize=figsize, dpi=DPI)

    title = f"Custom grid - {custom_bin}"
    
    sdata.pl.render_shapes(
            element=f"{custom_bin}_shapes", fill_alpha=0.15, outline_width=1, outline_alpha=1, color='#41b8de', outline_color='#41b8de'
    ).pl.show(
        coordinate_systems=dataset_id,
        title=title,
        figsize=figsize,
        colorbar=False,
        dpi=DPI,
        ax=ax,
        pad_extent=rel_fontsize(fig, 0.05)
    )

    ax.set_title(title, fontsize=rel_fontsize(fig, 0.02))
    ax.set_xlabel("x", fontsize=rel_fontsize(fig, 0.02))
    ax.set_ylabel("y", fontsize=rel_fontsize(fig, 0.02))
    ax.tick_params(axis="both", which="both", labelsize=rel_fontsize(fig, 0.015))

    save_path = os.path.join(level_plots_path, f'{custom_bin}.png')
    fig.savefig(save_path, bbox_inches="tight", pad_inches=max(fig.get_size_inches())/100, facecolor="white", transparent=False)

    plt.close(fig)

    print("Plot saved at:", save_path, flush=True)

    # Plot crop
    fig, ax = plt.subplots(figsize=figsize_crop, dpi=DPI)

    title = f"Custom grid - {custom_bin}"
    
    sdata_crop.pl.render_shapes(
            element=f"{custom_bin}_shapes", fill_alpha=0.15, outline_width=0.25, outline_alpha=1, color='#41b8de', outline_color='#41b8de', method="matplotlib"
    ).pl.show(
        coordinate_systems=dataset_id,
        title=title,
        figsize=figsize,
        colorbar=False,
        dpi=DPI,
        ax=ax,
    )

    ax.set_title(title, fontsize=rel_fontsize(fig, 0.02))
    ax.set_xlabel("x", fontsize=rel_fontsize(fig, 0.02))
    ax.set_ylabel("y", fontsize=rel_fontsize(fig, 0.02))
    ax.tick_params(axis="both", which="both", labelsize=rel_fontsize(fig, 0.015))

    ax.set_xlim(min_x, max_x)
    ax.set_ylim(min_y, max_y)
    ax.invert_yaxis()

    save_path = os.path.join(level_plots_path, f'crop_{custom_bin}.png')
    fig.savefig(save_path, bbox_inches="tight", pad_inches=max(fig.get_size_inches())/100, facecolor="white", transparent=False)

    plt.close(fig)

    print("Plot saved at:", save_path, flush=True)

# NOTE: Spots are represented true to scale

INFO     Value for parameter 'color' appears to be a color, using it as such.                                      
INFO     Using 'datashader' backend with 'None' as reduction method to speed up plotting. Depending on the         
         reduction method, the value range of the plot might change. Set method to 'matplotlib' to disable this    
         behaviour.                                                                                                
Plot saved at: /data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/Curio_seeker/SC14/harpy/plots/hexagonal_grid_20um/hexagonal_grid_20um.png
INFO     Value for parameter 'color' appears to be a color, using it as such.                                      
Plot saved at: /data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/Curio_seeker/SC14/harpy/plots/hexagonal_grid_20um/crop_hexagonal_grid_20um.png
INFO     Value for parameter 'color' appears to be a co

## Create bounding box for total capture area

In [24]:
# Add bounding box for total capture area
union_geom = sdata.shapes[dataset_id].union_all()
hull = union_geom.convex_hull

sdata = hp.sh.add_shapes_layer(
    sdata,
    input = gpd.GeoDataFrame(geometry=[hull]),
    output_layer = f'{dataset_id}_capture_area',
    transformations = get_transformation(sdata.shapes[dataset_id], get_all=True),
    overwrite = True,
)

In [26]:
# Plot capture area hull
fig, ax = plt.subplots(figsize=figsize, dpi=DPI)

title = "Capture area convex hull"

sdata.pl.render_shapes(
    element=f'{dataset_id}_capture_area', color='red', fill_alpha=0.05, outline_width=rel_fontsize(fig, 0.002), outline_alpha=1, outline_color='red'
).pl.show(
    coordinate_systems=dataset_id,
    title=title,
    figsize=figsize,
    colorbar=False,
    dpi=DPI,
    ax=ax,
    pad_extent=rel_fontsize(fig, 0.05)
)

ax.set_title(title, fontsize=rel_fontsize(fig, 0.02))
ax.set_xlabel("x", fontsize=rel_fontsize(fig, 0.02))
ax.set_ylabel("y", fontsize=rel_fontsize(fig, 0.02))
ax.tick_params(axis="both", which="both", labelsize=rel_fontsize(fig, 0.015))

save_path = os.path.join(plots_path, 'capture_area_hull.png')
fig.savefig(save_path, bbox_inches="tight", pad_inches=max(fig.get_size_inches())/100, facecolor="white", transparent=False)

plt.close(fig)

print("Plot saved at:", save_path)

INFO     Value for parameter 'color' appears to be a color, using it as such.                                      
Plot saved at: /data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/Curio_seeker/SC14/harpy/plots/capture_area_hull.png


In [27]:
# Plot capture area hull
for level in ['table']:
    gc.collect()
    
    level_plots_path = os.path.join(plots_path, level)
    os.makedirs(level_plots_path, exist_ok=True)

    fig, ax = plt.subplots(figsize=figsize, dpi=DPI)

    title = f"{pipeline_name} - Capture area hull"
    
    sdata.pl.render_shapes(
        element=f'{dataset_id}_capture_area', color='red', fill_alpha=0.05, outline_width=rel_fontsize(fig, 0.002), outline_alpha=1, outline_color='red'
    ).pl.render_shapes(
        element=dataset_id, fill_alpha=0, outline_width=1, outline_alpha=1, outline_color='#41b8de'
    ).pl.show(
        coordinate_systems=dataset_id,
        title=title,
        figsize=figsize,
        colorbar=False,
        dpi=DPI,
        ax=ax,
        pad_extent=rel_fontsize(fig, 0.05)
    )

    ax.set_title(title, fontsize=rel_fontsize(fig, 0.02))
    ax.set_xlabel("x", fontsize=rel_fontsize(fig, 0.02))
    ax.set_ylabel("y", fontsize=rel_fontsize(fig, 0.02))
    ax.tick_params(axis="both", which="both", labelsize=rel_fontsize(fig, 0.015))

    save_path = os.path.join(level_plots_path, 'capture_area_hull.png')
    fig.savefig(save_path, bbox_inches="tight", pad_inches=max(fig.get_size_inches())/100, facecolor="white", transparent=False)

    plt.close(fig)

    print("Plot saved at:", save_path, flush=True)

INFO     Value for parameter 'color' appears to be a color, using it as such.                                      
INFO     Using 'datashader' backend with 'None' as reduction method to speed up plotting. Depending on the         
         reduction method, the value range of the plot might change. Set method to 'matplotlib' to disable this    
         behaviour.                                                                                                
Plot saved at: /data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/Curio_seeker/SC14/harpy/plots/table/capture_area_hull.png


In [28]:
# Area of capture area in mm²
gdf = sdata.shapes[f'{dataset_id}_capture_area']
capture_area = (gdf.geometry.area.sum() * microns_per_pixel**2) / 1000000
print(f"Area of capture area: {capture_area:.2f} mm²")

# NOTE: This is somewhat biased because we are using the size of the capture area to determine the pixel size. However, it should be close.

Area of capture area: 8.53 mm²


In [29]:
from math import dist
coords = list(bbox.exterior.coords)

# Compute lengths of bbox
side1 = dist(coords[0], coords[1]) * microns_per_pixel / 10000
side2 = dist(coords[1], coords[2]) * microns_per_pixel / 10000

print(f"Dimensions of capture area bbox: {side1:.2f} cm x {side2:.2f} cm")

# NOTE: This is biased because we are using the size of the capture area to determine the pixel size

Dimensions of capture area bbox: 0.30 cm x 0.30 cm


## QC

In [30]:
# Calculate QC metrics using scanpy's calculate_qc_metrics()
for level in levels:
    sdata = qc_metrics(
        sdata = sdata,
        table_layer = level,
        species = species
    )

for custom_bin in custom_bins:
    sdata = qc_metrics(
        sdata = sdata,
        table_layer = f"{custom_bin}_table",
        species = species
    )

2026-01-28 12:10:56.499 | WARNING  | harpy.utils._io:_incremental_io_on_disk:53 - layer with name 'table' already exists. Overwriting...
2026-01-28 12:11:06.688 | WARNING  | harpy.utils._io:_incremental_io_on_disk:53 - layer with name 'hexagonal_grid_20um_table' already exists. Overwriting...
2026-01-28 12:11:13.610 | WARNING  | harpy.utils._io:_incremental_io_on_disk:53 - layer with name 'hexagonal_grid_120um_table' already exists. Overwriting...


In [31]:
# Plot QC metrics
for level in levels:
    for qc_metric in ['n_genes_by_counts', 'total_counts', 'pct_counts_in_top_50_genes', 'total_counts_mt', 'pct_counts_mt']:
        gc.collect()

        level_plots_path = os.path.join(plots_path, level)
        os.makedirs(level_plots_path, exist_ok=True)

        values = sdata.tables[level].obs[qc_metric].to_numpy()
        vmin, vmax = np.percentile(values, [1, 99])
        norm = matplotlib.colors.Normalize(vmin=vmin, vmax=vmax)
        
        fig, ax = plt.subplots(figsize=figsize, dpi=DPI)

        title = f"Scanpy QC - {qc_metric}"

        sdata.pl.render_labels(
            element=f"{dataset_id}_labels", fill_alpha=1, contour_px=5, outline_alpha=0, color=qc_metric, table_name=level, cmap="viridis", norm=norm
        ).pl.show(
            coordinate_systems=dataset_id,
            title=title,
            figsize=figsize,
            colorbar=True,
            dpi=DPI,
            ax=ax,
            pad_extent=rel_fontsize(fig, 0.05)
        )

        ax.set_title(title, fontsize=rel_fontsize(fig, 0.02))
        ax.set_xlabel("x", fontsize=rel_fontsize(fig, 0.02))
        ax.set_ylabel("y", fontsize=rel_fontsize(fig, 0.02))
        ax.tick_params(axis="both", which="both", labelsize=rel_fontsize(fig, 0.015))

        cbar = fig.axes[-1]
        cbar.set_ylabel(qc_metric, rotation=270, labelpad=rel_fontsize(fig, 0.025), fontsize = rel_fontsize(fig, 0.02))
        cbar.tick_params(axis="y", which="both", labelsize=rel_fontsize(fig, 0.015))

        save_path = os.path.join(level_plots_path, f'Scanpy_QC_{qc_metric}.png')
        fig.savefig(save_path, bbox_inches="tight", pad_inches=max(fig.get_size_inches())/100, facecolor="white", transparent=False)

        plt.close(fig)

        print("Plot saved at:", save_path, flush=True)

Plot saved at: /data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/Curio_seeker/SC14/harpy/plots/table/Scanpy_QC_n_genes_by_counts.png
Plot saved at: /data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/Curio_seeker/SC14/harpy/plots/table/Scanpy_QC_total_counts.png
Plot saved at: /data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/Curio_seeker/SC14/harpy/plots/table/Scanpy_QC_pct_counts_in_top_50_genes.png
Plot saved at: /data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/Curio_seeker/SC14/harpy/plots/table/Scanpy_QC_total_counts_mt.png
Plot saved at: /data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/Curio_seeker/SC14/harpy/plots/table/Scanpy_QC_pct_counts_mt.png


In [32]:
# Plot QC metrics
for custom_bin in custom_bins:
    for qc_metric in ['n_genes_by_counts', 'total_counts', 'pct_counts_in_top_50_genes', 'total_counts_mt', 'pct_counts_mt']:
        gc.collect()

        level_plots_path = os.path.join(plots_path, custom_bin)
        os.makedirs(level_plots_path, exist_ok=True)

        values = sdata.tables[f"{custom_bin}_table"].obs[qc_metric].to_numpy()
        vmin, vmax = np.percentile(values, [1, 99])
        norm = matplotlib.colors.Normalize(vmin=vmin, vmax=vmax)
        
        fig, ax = plt.subplots(figsize=figsize, dpi=DPI)

        title = f"Scanpy QC - {qc_metric}"

        sdata.pl.render_labels(
            element=f"{custom_bin}_labels", fill_alpha=1, contour_px=5, outline_alpha=0, color=qc_metric, table_name=f"{custom_bin}_table", cmap="viridis", norm=norm
        ).pl.show(
            coordinate_systems=dataset_id,
            title=title,
            figsize=figsize,
            colorbar=True,
            dpi=DPI,
            ax=ax,
            pad_extent=rel_fontsize(fig, 0.05)
        )

        ax.set_title(title, fontsize=rel_fontsize(fig, 0.02))
        ax.set_xlabel("x", fontsize=rel_fontsize(fig, 0.02))
        ax.set_ylabel("y", fontsize=rel_fontsize(fig, 0.02))
        ax.tick_params(axis="both", which="both", labelsize=rel_fontsize(fig, 0.015))

        cbar = fig.axes[-1]
        cbar.set_ylabel(qc_metric, rotation=270, labelpad=rel_fontsize(fig, 0.025), fontsize = rel_fontsize(fig, 0.02))
        cbar.tick_params(axis="y", which="both", labelsize=rel_fontsize(fig, 0.015))

        save_path = os.path.join(level_plots_path, f'Scanpy_QC_{qc_metric}.png')
        fig.savefig(save_path, bbox_inches="tight", pad_inches=max(fig.get_size_inches())/100, facecolor="white", transparent=False)

        plt.close(fig)

        print("Plot saved at:", save_path, flush=True)

Plot saved at: /data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/Curio_seeker/SC14/harpy/plots/hexagonal_grid_20um/Scanpy_QC_n_genes_by_counts.png
Plot saved at: /data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/Curio_seeker/SC14/harpy/plots/hexagonal_grid_20um/Scanpy_QC_total_counts.png
Plot saved at: /data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/Curio_seeker/SC14/harpy/plots/hexagonal_grid_20um/Scanpy_QC_pct_counts_in_top_50_genes.png
Plot saved at: /data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/Curio_seeker/SC14/harpy/plots/hexagonal_grid_20um/Scanpy_QC_total_counts_mt.png
Plot saved at: /data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/Curio_seeker/SC14/harpy/plots/hexagonal_grid_20um/Scanpy_QC_pct_counts_mt.png
Plot saved at: /data/groups/technologies/spatial.catalyst/Proj

## Leiden clustering

In [84]:
from harpy.utils._keys import _INSTANCE_KEY, _REGION_KEY

for custom_bin in custom_bins:
    gc.collect()
    
    print(f'running: {custom_bin}', flush=True)

    labels_layer=f"{custom_bin}_labels"
    shapes_layer=f"{custom_bin}_shapes"
    table_layer=f"{custom_bin}_table"
    
    level_plots_path = os.path.join(plots_path, custom_bin)
    os.makedirs(level_plots_path, exist_ok=True)
    
    # Only keep spots in tissue for clustering
    adata = sdata.tables[table_layer]
    
    sdata = hp.tb.add_table_layer(
        sdata,
        adata=adata.copy(),
        output_layer=f"{table_layer}_processed",
        region=adata.obs[_REGION_KEY].cat.categories.to_list(),
        overwrite=True,
    )
    
    # Preprocess data
    sdata = hp.tb.preprocess_transcriptomics(
        sdata=sdata,
        size_norm=False,
        labels_layer=labels_layer,
        table_layer=f"{table_layer}_processed",
        output_layer=f"{table_layer}_processed",
        min_counts=10,
        min_cells=5,
        highly_variable_genes=True,
        max_value_scale=10,
        n_comps=50,
        update_shapes_layers=False,
        overwrite=True
    )
    
    gc.collect()
    
    # Leiden clustering
    leiden_resolutions = [0.75, 1, 1.25]
    calculate_umap = True
    calculate_neighbors = True

    for res in leiden_resolutions:
        gc.collect()

        res_str = str(res).replace('.', '')
        print(f'Leiden resolution: {res}', flush=True)
                
        sdata = hp.tb.leiden(
            sdata, 
            labels_layer=labels_layer,
            table_layer=f"{table_layer}_processed",
            output_layer=f"{table_layer}_processed",
            calculate_umap=calculate_umap,
            calculate_neighbors=calculate_neighbors,
            rank_genes=False,
            n_neighbors=35,
            n_pcs=30,
            resolution=res,
            key_added=f'leiden_{res_str}',
            index_names_var=None,
            index_positions_var=None,
            random_state=100,
            overwrite=True
        )
        
        calculate_umap = False # Only calculate UMAP once
        calculate_neighbors = False # Only calculate neighbors once

        # Plot Leiden clustering
        gc.collect()
        
        with matplotlib.rc_context({'legend.markerscale': 2}):
            fig, ax = plt.subplots(figsize=figsize, dpi=DPI)

            title = f"Leiden {res}"
            
            # NOTE: Since there are >100 clusters for SC14 for the 20 um hexagonal grid, the number of Leiden clusters that is plotted needs to be capped. Otherwise, spatialdata-plot gives an error.
            vc = sdata.tables[f"{table_layer}_processed"].obs[f'leiden_{res_str}'].value_counts()
            keep = vc.nlargest(98).index

            sdata.tables[f"{table_layer}_processed"].obs[f"leiden_{res_str}"] = sdata.tables[f"{table_layer}_processed"].obs[f"leiden_{res_str}"].cat.add_categories(["other"])
            sdata.tables[f"{table_layer}_processed"].obs[f"leiden_{res_str}_capped"] = sdata.tables[f"{table_layer}_processed"].obs[f"leiden_{res_str}"].where(
                sdata.tables[f"{table_layer}_processed"].obs[f"leiden_{res_str}"].isin(keep), other="other"
            )
            sdata.tables[f"{table_layer}_processed"].obs[f"leiden_{res_str}_capped"]=sdata.tables[f"{table_layer}_processed"].obs[f"leiden_{res_str}_capped"].cat.remove_unused_categories()
            
            cmap = glasbey_cmap(sdata.tables[f"{table_layer}_processed"].obs[f"leiden_{res_str}_capped"].nunique())
            
            sdata.pl.render_labels(
                element=labels_layer, fill_alpha=1, contour_px=5, outline_alpha=0, cmap=cmap, color=f"leiden_{res_str}_capped", table_name=f"{table_layer}_processed",
            ).pl.show(
                coordinate_systems=dataset_id,
                title=title,
                figsize=figsize,
                colorbar=False,
                legend_fontsize=rel_fontsize(fig, 0.015), 
                legend_fontweight='bold', 
                legend_loc='right margin',
                na_in_legend=False,
                dpi=DPI,
                ax=ax,
            )

            ax.set_title(title, fontsize=rel_fontsize(fig, 0.02))
            ax.set_xlabel("x", fontsize=rel_fontsize(fig, 0.02))
            ax.set_ylabel("y", fontsize=rel_fontsize(fig, 0.02))
            ax.tick_params(axis="both", which="both", labelsize=rel_fontsize(fig, 0.015))

            save_path = os.path.join(level_plots_path, f'Leiden_{res}.png')
            fig.savefig(save_path, bbox_inches="tight", pad_inches=max(fig.get_size_inches())/100, facecolor="white", transparent=False)

            plt.close(fig)

            print("Plot saved at:", save_path, flush=True)
            
        from matplotlib.colors import to_hex
        colors = [to_hex(c) for c in cmap.colors]
        sdata.tables[f"{table_layer}_processed"].uns[f"leiden_{res_str}_capped_colors"] = colors
            
        hp.pl.cluster(
            sdata, 
            table_layer = f"{table_layer}_processed",
            key_added=f"leiden_{res_str}_capped",
            output=os.path.join(level_plots_path, f'Leiden_{res}')
        )

running: hexagonal_grid_20um


2026-01-28 14:32:12.700 | WARNING  | harpy.utils._io:_incremental_io_on_disk:53 - layer with name 'hexagonal_grid_20um_table_processed' already exists. Overwriting...
2026-01-28 14:32:18.372 | INFO     | harpy.table._preprocess:preprocess:314 - Calculating cell size from provided labels_layer 'hexagonal_grid_20um_labels'
2026-01-28 14:32:19.990 | INFO     | harpy.table._preprocess:preprocess:334 - Saving non preprocessed data matrix to the AnnData '.layers[raw_counts]'.
2026-01-28 14:32:20.130 | WARNING  | harpy.table._preprocess:preprocess:395 - amount of pc's was set to 15 because of the dimensionality of 'sdata.tables[table_layer]'.
2026-01-28 14:32:27.824 | WARNING  | harpy.utils._io:_incremental_io_on_disk:53 - layer with name 'hexagonal_grid_20um_table_processed' already exists. Overwriting...


Leiden resolution: 0.75


2026-01-28 14:35:39.531 | WARNING  | harpy.utils._io:_incremental_io_on_disk:53 - layer with name 'hexagonal_grid_20um_table_processed' already exists. Overwriting...


Plot saved at: /data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/Curio_seeker/SC14/harpy/plots/hexagonal_grid_20um/Leiden_0.75.png
Leiden resolution: 1


...

Plot saved at: /data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/Curio_seeker/SC14/harpy/plots/hexagonal_grid_120um/Leiden_0.75.png
Leiden resolution: 1


2026-01-28 14:39:50.320 | WARNING  | harpy.utils._io:_incremental_io_on_disk:53 - layer with name 'hexagonal_grid_120um_table_processed' already exists. Overwriting...


Plot saved at: /data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/Curio_seeker/SC14/harpy/plots/hexagonal_grid_120um/Leiden_1.png
Leiden resolution: 1.25


2026-01-28 14:40:41.791 | WARNING  | harpy.utils._io:_incremental_io_on_disk:53 - layer with name 'hexagonal_grid_120um_table_processed' already exists. Overwriting...


Plot saved at: /data/groups/technologies/spatial.catalyst/Projects/2024-07-UTBenchmark-SpC/data/processed/Curio_seeker/SC14/harpy/plots/hexagonal_grid_120um/Leiden_1.25.png


## Spots metrics

In [42]:
gc.collect()

7830

In [43]:
metrics = pd.read_csv(os.path.join(input_path, f'{dataset_id}_Metrics.csv'))
metrics

,Unnamed: 0,Values,Category
0,Total_raw_fastq_readpairs,1.609948e+08,read_quality
1,Readpairs_proper_structure,1.140148e+08,read_quality
2,Readpairs_with_adapter,2.561578e+06,read_quality
3,Percent_readpairs_proper_structure,7.082000e+01,read_quality
4,Percent_readpairs_with_adapter,1.590000e+00,read_quality
5,Percent_adapter_free_readpairs_proper_structure,7.196000e+01,read_quality
6,Total_reads_in_genes,1.859422e+07,read_quality
7,Percent_reads_in_genes,1.155000e+01,read_quality
8,Total_barcodes_on_substrate,7.122300e+04,bead_matching_quality
9,Total_barcodes_recovered,2.690100e+04,bead_matching_quality


In [44]:
# Total number of spots in capture area (according to metrics overview)
n_spots_capture_area = metrics.loc[metrics["Unnamed: 0"] == "Total_barcodes_on_substrate", "Values"].iloc[0]
print(f"Total number of spots in capture area: {n_spots_capture_area:,.0f}")

Total number of spots in capture area: 71,223


In [45]:
# Total number of recovered (according to metrics overview)
n_spots_recovered = metrics.loc[metrics["Unnamed: 0"] == "Total_barcodes_recovered", "Values"].iloc[0]
print(f"Total number of spots recovered: {n_spots_recovered:,.0f}")

Total number of spots recovered: 26,901


In [46]:
# Total barcodes in barcode decoding file
barcodes_whitelist = pd.read_csv(barcode_decoding_file, sep='\s+', header=None, names=['barcode', 'x', 'y'])
print(f"Total number of spots in decoding file: {len(barcodes_whitelist['barcode']):,}")

Total number of spots in decoding file: 71,223


In [47]:
# Total number of spots in table
n_spots = len(sdata.tables['table'].obs)
print(f"Number of spots in table: {n_spots:,}")

Number of spots in table: 26,901


In [48]:
# Save barcodes
with open(os.path.join(os.path.dirname(output_path), "all_barcodes.txt"), "w") as f:
    for item in barcodes_whitelist['barcode']:
        f.write(f"{item}\n")
        
with open(os.path.join(os.path.dirname(output_path), "barcodes_in_tissue.txt"), "w") as f:
    for item in sdata.tables['table'].obs.index:
        f.write(f"{item}\n")      

## Capture effeciency metrics

In [49]:
gc.collect()

0

In [50]:
def row_sums(X):
    """Return 1D array of row sums without densifying."""
    return np.asarray(X.sum(axis=1)).ravel()

def col_sums(X):
    """Return 1D array of column sums without densifying."""
    return np.asarray(X.sum(axis=0)).ravel()

In [51]:
# Define Anndata
adata = sdata.tables['table']

In [52]:
# Total number of spots in table with at least one UMI
n_spots_not_zero = (row_sums(adata.X) > 0).sum()
print(f"Number of spots in table with at least one UMI: {n_spots_not_zero:,}")

Number of spots in table with at least one UMI: 26,901


In [53]:
# Total number of genes in table
print(f"Total number of genes in table: {(col_sums(adata.X) > 0).sum():,}")

Total number of genes in table: 22,303


In [54]:
# Total number of unique UMIs
print(f"Total number of unique UMIs: {adata.X.sum():,.0f}")

Total number of unique UMIs: 579,472


In [55]:
# Total number of unique UMIs
print(f"Total number of unique UMIs: {sdata.tables['table'].obs['nCount_RNA'].sum():,.0f}")

Total number of unique UMIs: 579,472


In [56]:
# Total number of UMIs
print(f"Total number of UMIs: {sdata.tables['table'].obs['numReads'].sum():,.0f}")

Total number of UMIs: 12,360,822
